In [42]:
import json
import os
import pickle
import re

# data preprocessing
from collections import defaultdict
from io import StringIO
from itertools import combinations
from urllib.parse import quote

import matplotlib.pyplot as plt

# networks
import networkx as nx

# from networkx.readwrite import gpickle
import pandas as pd

# webscraping
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm

In [57]:
def get_character_pulls(version: int = 52) -> pd.DataFrame:
    """
    Fetches character data from the YSHelper API for a given game version,
    cleans character names, and calculates the estimated number of pulls
    based on ownership and constellation rates.

    Args:
        version (str): The game version to fetch data for (e.g., "4.1").

    Returns:
        pd.DataFrame: A DataFrame where each row corresponds to a character
        and columns include ownership, constellation rates, and estimated
        number of pulls.
    """
    url = f"https://api.yshelper.com/ys/getAbyssRank.php?star=all&role=all&lang=en&version={version}"
    data = requests.get(url).json()

    chars = {}
    for rank_char in data["result"][0]:
        for char in rank_char["list"]:
            char_name = char["name"]
            chars[char_name] = char

    df_owns = pd.DataFrame(chars).T

    # Clean character names
    df_owns["name"] = df_owns["name"].replace({"Ambor": "Amber", "Ayaka": "Kamisato Ayaka"})

    # Calculate number of pulls
    c_rate_cols = ["c0_rate", "c1_rate", "c2_rate", "c3_rate", "c4_rate", "c5_rate", "c6_rate"]
    df_owns.loc[:, c_rate_cols] = df_owns[c_rate_cols] / 100
    df_owns["pull_number"] = df_owns["own"] * sum(df_owns[c] * (i + 1) for i, c in enumerate(c_rate_cols))
    df_owns["pull_number"] = df_owns["pull_number"].astype(int)

    return df_owns[['name', 

In [63]:
get_character_pulls()[['name', 'star', 'use', 'own', 'pull_number']]

,name,star,use,own,pull_number
Lauma,Lauma,5,31950,34119,43433
Nefer,Nefer,5,23709,27285,38035
Citlali,Citlali,5,45507,59631,90042
Bennett,Bennett,4,57123,77430,497023
Xilonen,Xilonen,5,46416,63108,85637
...,...,...,...,...,...
Sethos,Sethos,4,12,76470,516172
Xinyan,Xinyan,4,9,74130,74130
Chongyun,Chongyun,4,45,75594,408434
Razor,Razor,4,18,73572,380367
